# FMP 재무데이터 분석 v1 (US_IS/BS/CF_from_FMP)

`fmp_fs_analyzer_v1.py` 사용. DART 분석(`dart_fs_analyzer_v4`)과 같은 인터페이스.

**DART 와 다른 점**
- 단위 USD (`unit=1e6` → $M)
- FMP 분기값은 이미 분기 flow → 분기화 없음. `period`(회계분기) 대신 `date`(분기말)를 달력 분기로 변환해 정렬
- 저장 라이브러리(V4)가 단순 INSERT 라 같은 (ticker, date, item) 이 여러 번 쌓임 → **id 최대 행(최신 수집분)만 사용**
- 기업명 없음 (`name_table` 로 선택 매핑)

**데이터 업데이트**: `python US_FMP_FS_1_RUN_UPDATE.py` (2번 파일은 import 전용). 중복이 쌓이면 `python US_FMP_FS_2_DB_SAVE_LIB.py --truncate` 후 `--quarters 60` 재적재.

In [1]:
# ==========================================================
# PART 1: 환경 + Control Panel + DB
# ==========================================================
import sys
from pathlib import Path
import pandas as pd

def add_repo_path():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'DATA').exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            return parent
    raise FileNotFoundError('DATA 폴더를 찾을 수 없습니다')

PROJECT_ROOT = add_repo_path()
sys.path.insert(0, str(Path.cwd()))

from DATA import config
import FMP_fs_analyzer_v1 as F

engine = config.get_engine(config.get_db_info())

# ---- Control Panel ----
ASOF        = None          # 기준 분기 '2026Q2'. None → 최근 완전 적재 분기 자동
START       = '2019-01-01'  # 패널 시작일 (TTM/평균용 여유 포함)
TICKERS     = None          # None=전체, 또는 ['AAPL','MSFT']
NAME_TABLE  = None          # 기업명 테이블 (예: 'us_ticker_master'); 없으면 None
TOP_N       = 100
UNIT        = 1e6           # USD → $M
PANEL_CACHE = Path('fmp_panel_cache.parquet')   # None 이면 캐시 안 씀. 데이터 재적재 후에는 삭제!

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 0. 적재 상태 점검
`dup_factor` 가 2~3 이상이면 중복이 많이 쌓인 것 → `--truncate` 후 재적재 권장 (분석은 최신 행만 쓰므로 결과에는 영향 없음, 속도만 느려짐).

In [17]:
F.duplicate_report(engine)

,table,total_rows,unique_keys,dup_factor,n_ticker,max_date
0,US_IS_from_FMP,4115776,3266508,1.26,1989,2026-08-02
1,US_BS_from_FMP,6445868,5110776,1.26,1990,2026-08-02
2,US_CF_from_FMP,4398630,3489240,1.26,1989,2026-08-02


## 1. 적재된 항목 목록 (키워드 검색)
`mapped` = concept 매핑. 빈칸인 항목이 필요하면 `F.CONCEPTS` 에 추가.

In [3]:
F.search_items(engine, 'income')
# F.search_items(engine, sj_div='BS')
# F.search_items(engine)                       # 전체

,mapped,sj_div,item,n_ticker,n_rows,min_date,max_date
0,세전이익,IS,incomeBeforeTax,1989,146992,1996-03-31,2026-08-02
1,,IS,incomeBeforeTaxRatio,1989,146992,1996-03-31,2026-08-02
2,법인세,IS,incomeTaxExpense,1989,146992,1996-03-31,2026-08-02
3,,IS,interestIncome,1989,146992,1996-03-31,2026-08-02
4,당기순이익,IS,netIncome,1989,146992,1996-03-31,2026-08-02
5,,IS,netIncomeRatio,1989,146992,1996-03-31,2026-08-02
6,영업이익,IS,operatingIncome,1989,146992,1996-03-31,2026-08-02
7,,IS,operatingIncomeRatio,1989,146992,1996-03-31,2026-08-02
8,,IS,totalOtherIncomeExpensesNet,1989,146992,1996-03-31,2026-08-02
9,,BS,accumulatedOtherComprehensiveIncomeLoss,1990,146497,1987-12-31,2026-08-02


## 2. 패널 구축 (1회)

In [4]:
if PANEL_CACHE and PANEL_CACHE.exists() and TICKERS is None:
    panel = pd.read_parquet(PANEL_CACHE)
    panel['q'] = pd.PeriodIndex(panel['q'], freq='Q')
    print(f'cache load: {PANEL_CACHE} ({len(panel):,} rows)')
else:
    panel = F.build_panel(engine, tickers=TICKERS, start=START, name_table=NAME_TABLE)
    if PANEL_CACHE and TICKERS is None:
        panel.assign(q=panel['q'].astype(str)).to_parquet(PANEL_CACHE, index=False)
        print(f'cache save: {PANEL_CACHE}')
print(panel.shape, panel['ticker'].nunique(), 'tickers')
F.coverage_report(panel)

cache load: fmp_panel_cache.parquet (2,204,476 rows)
(2204476, 8) 1988 tickers


,n_ticker,n_rows,min_q,max_q
concept,,,,
현금성자산,1988,55120,2019Q1,2026Q3
유동부채,1988,55120,2019Q1,2026Q3
유형자산,1988,55120,2019Q1,2026Q3
자본,1988,55120,2019Q1,2026Q3
순차입금,1988,55120,2019Q1,2026Q3
비지배지분,1988,55120,2019Q1,2026Q3
부채,1988,55120,2019Q1,2026Q3
자산,1988,55120,2019Q1,2026Q3
장기차입금,1988,55120,2019Q1,2026Q3


## 3. 종목별 시계열
`ts.attrs['fiscal_dates']` 에 달력 분기 ↔ 실제 회계분기말 매핑.

In [5]:
ts = F.get_ts(panel, 'AAPL', ['매출액', '매출총이익', '영업이익', '당기순이익', 'EPS', '자본', '영업현금흐름', 'FCF'], start='2023Q1', unit=UNIT)
print(ts.attrs['ticker'], ts.attrs['fiscal_dates'].get(ts.index[-1]))
ts

AAPL 2026-06-27


concept,매출액,매출총이익,영업이익,당기순이익,EPS,자본,영업현금흐름,FCF
q,,,,,,,,
2023Q2,"94,836.00","41,976.00","28,318.00","24,160.00",1.52,"62,158.00","28,560.00","25,644.00"
2023Q3,"81,797.00","36,413.00","22,998.00","19,881.00",1.26,"60,274.00","26,380.00","24,287.00"
2023Q4,"119,575.00","54,855.00","40,373.00","33,916.00",2.18,"74,100.00","39,895.00","37,503.00"
2024Q1,"90,753.00","42,271.00","27,900.00","23,636.00",1.53,"74,194.00","22,690.00","20,694.00"
2024Q2,"85,777.00","39,678.00","25,352.00","21,448.00",1.40,"66,708.00","28,858.00","26,707.00"
2024Q3,"94,930.00","43,879.00","29,591.00","14,736.00",0.97,"56,950.00","26,811.00","23,903.00"
2024Q4,"124,300.00","58,275.00","42,832.00","36,330.00",2.40,"66,758.00","29,935.00","26,995.00"
2025Q1,"95,359.00","44,867.00","29,589.00","24,780.00",1.65,"66,796.00","23,952.00","20,881.00"
2025Q2,"94,036.00","43,718.00","28,202.00","23,434.00",1.57,"65,830.00","27,867.00","24,405.00"


## 4. YoY / QoQ 상위 N
`min_base` USD: 기준값 하한 (1e7 = $10M).

In [6]:
F.yoy_screen(panel, '매출액', n=TOP_N, asof=ASOF, unit=UNIT, min_base=1e7)

,ticker,company_name,base_q,t_q,매출액(t-4),매출액(t),growth_%
0,YPF,,2025Q1,2026Q1,"4,600.00","6,956,434.00","151,126.83"
1,QXO,,2025Q1,2026Q1,13.51,"1,730.20","12,708.71"
2,MDC,,2025Q1,2026Q1,24.77,"1,209.07","4,780.78"
3,NRZ,,2025Q1,2026Q1,28.89,375.06,"1,198.46"
4,GMAB,,2025Q1,2026Q1,105.23,899.32,754.59
...,...,...,...,...,...,...,...
95,HL,,2025Q1,2026Q1,261.34,411.43,57.43
96,AXSM,,2025Q1,2026Q1,121.46,191.20,57.42
97,DHT,,2025Q1,2026Q1,118.57,186.48,57.27
98,FIX,,2025Q1,2026Q1,"1,831.29","2,865.33",56.47


In [7]:
F.yoy_screen(panel, '영업이익', n=TOP_N, asof=ASOF, unit=UNIT, min_base=5e6)

,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),growth_%
0,YPF,,2025Q1,2026Q1,412.00,"1,241,859.00","301,322.09"
1,HOV,,2025Q1,2026Q1,19.55,562.66,"2,777.34"
2,TPX,,2025Q1,2026Q1,13.20,207.60,"1,472.73"
3,ALNY,,2025Q1,2026Q1,18.08,268.64,"1,386.07"
4,F,,2025Q1,2026Q1,164.00,"2,329.00","1,320.12"
...,...,...,...,...,...,...,...
95,MUSA,,2025Q1,2026Q1,88.30,204.90,132.05
96,UNFI,,2025Q1,2026Q1,41.00,95.00,131.71
97,AEIS,,2025Q1,2026Q1,30.60,70.90,131.70
98,WPM,,2025Q1,2026Q1,290.68,666.92,129.43


In [8]:
F.qoq_screen(panel, '매출액', n=TOP_N, asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,매출액(t-1),매출액(t),growth_%
0,TKC,,2025Q4,2026Q1,"1,662.46","68,376.96","4,012.99"
1,KOF,,2025Q4,2026Q1,"4,248.02","70,926.00","1,569.62"
2,HRB,,2025Q4,2026Q1,198.87,"2,398.11","1,105.90"
3,PAC,,2025Q4,2026Q1,"1,012.15","11,369.63","1,023.32"
4,WBI,,2025Q4,2026Q1,27.43,200.98,632.77
...,...,...,...,...,...,...,...
95,OLLI,,2025Q4,2026Q1,613.62,779.26,26.99
96,CDE,,2025Q4,2026Q1,674.85,856.19,26.87
97,BBW,,2025Q4,2026Q1,122.68,154.51,25.95
98,FSM,,2025Q4,2026Q1,272.03,342.47,25.90


In [9]:
F.qoq_screen(panel, '영업이익', n=TOP_N, asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-1),영업이익(t),growth_%
0,TKC,,2025Q4,2026Q1,260.30,"10,798.52","4,048.48"
1,UNH,,2025Q4,2026Q1,380.00,"8,990.00","2,265.79"
2,GLP,,2025Q4,2026Q1,16.43,352.10,"2,042.92"
3,SKM,,2025Q4,2026Q1,"28,249.00","547,805.00","1,839.20"
4,M,,2025Q4,2026Q1,42.00,739.00,"1,659.52"
...,...,...,...,...,...,...,...
95,NVO,,2025Q4,2026Q1,"31,736.00","59,618.00",87.86
96,NAT,,2025Q4,2026Q1,20.43,38.36,87.74
97,RRC,,2025Q4,2026Q1,231.02,433.16,87.50
98,ALG,,2025Q4,2026Q1,22.53,42.16,87.12


## 5. 영업이익 흑자전환

In [10]:
F.turnaround_screen(panel, basis='yoy', asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-4),영업이익(t),swing,매출액(t),swing_%rev
0,TAK,,2025Q1,2026Q1,"-74,900.00","47,541.64","122,441.64","1,114,648.92",10.98
1,VOD,,2025Q1,2026Q1,"-2,793.00","1,128.35","3,921.35","21,139.62",18.55
2,APD,,2025Q1,2026Q1,"-2,328.00",752.70,"3,080.70","3,171.80",97.13
3,PSX,,2025Q1,2026Q1,-166.00,"2,854.00","3,020.00","34,076.00",8.86
4,VLO,,2025Q1,2026Q1,-900.00,"1,731.00","2,631.00","32,381.00",8.13
...,...,...,...,...,...,...,...,...,...
91,SBET,,2025Q1,2026Q1,-0.93,1.75,2.68,12.06,22.19
92,OOMA,,2025Q1,2026Q1,-0.32,2.10,2.43,74.58,3.25
93,ALLT,,2025Q1,2026Q1,-0.71,1.53,2.24,26.43,8.47
94,BLFS,,2025Q1,2026Q1,-1.22,0.03,1.25,27.50,4.53


In [11]:
F.turnaround_screen(panel, basis='qoq', asof=ASOF, unit=UNIT)

,ticker,company_name,base_q,t_q,영업이익(t-1),영업이익(t),swing,매출액(t),swing_%rev
0,TEO,,2025Q4,2026Q1,"-864,362.00","823,750.00","1,688,112.00","2,357,686.00",71.60
1,CYH,,2025Q4,2026Q1,"-10,954.00","1,644.00","12,598.00","12,158.00",103.62
2,JD,,2025Q4,2026Q1,"-5,849.00","3,783.97","9,632.98","313,784.66",3.07
3,GM,,2025Q4,2026Q1,"-3,647.00","2,926.00","6,573.00","43,624.00",15.07
4,CNC,,2025Q4,2026Q1,"-1,740.38","1,861.00","3,601.38","49,944.00",7.21
...,...,...,...,...,...,...,...,...,...
91,MRTN,,2025Q4,2026Q1,-7.49,0.17,7.66,203.53,3.76
92,SXC,,2025Q4,2026Q1,-2.10,4.40,6.50,455.10,1.43
93,MTRX,,2025Q4,2026Q1,-1.98,1.94,3.92,206.71,1.89
94,VECO,,2025Q4,2026Q1,-1.49,1.23,2.72,158.34,1.72


## 6. 재무비율 스크리너
`OPM, NPM, GPM, EBITDA_margin, ROE, ROE_지배, ROA, ROIC, 부채비율, 순차입금비율, OCF_margin, FCF_margin, FCF_conversion`

- 손익·CF TTM 합, BS 기초/기말 평균
- ROIC = OP×(1−유효세율)/(자본+순차입금) 평균. 순차입금은 FMP `netDebt` 우선, 없으면 totalDebt − 현금성자산
- 유효세율 = TTM 법인세/세전이익 (0~35% 클립, 산출 불가 시 21%)

In [12]:
ratios = F.compute_ratios(panel, asof=ASOF, ttm=True, unit=UNIT)
ratios.shape

(1988, 33)

In [13]:
F.ratio_screen(None, 'ROE',  n=TOP_N, ratios_df=ratios, unit=UNIT)

,ticker,company_name,t_q,당기순이익,자본(평균),ROE,OPM,NPM,ROA,ROIC,부채비율
0,MAS,,2026Q1,837.00,10.50,"7,971.43",16.79,10.90,16.19,33.27,"19,281.48"
1,CHH,,2026Q1,345.72,36.75,940.81,26.99,21.55,12.52,17.15,"2,042.65"
2,CLX,,2026Q1,756.00,141.00,536.17,15.68,11.18,12.65,26.23,"6,895.65"
3,COMM,,2026Q1,"7,006.80","1,645.25",425.88,-2.84,542.66,108.21,-0.76,18.47
4,BLKB,,2026Q1,141.79,33.86,418.69,19.52,12.44,6.73,16.01,"6,004.35"
...,...,...,...,...,...,...,...,...,...,...,...
95,ULTA,,2026Q1,"1,153.48","2,645.90",43.59,12.50,9.31,17.74,28.30,149.67
96,EBAY,,2026Q1,"2,040.00","4,681.00",43.58,19.58,17.58,11.07,22.94,305.55
97,GWW,,2026Q1,"1,782.00","4,095.00",43.52,14.23,9.70,19.66,31.56,118.12
98,GILD,,2026Q1,"9,216.00","21,254.50",43.36,38.26,30.99,16.35,26.38,140.19


In [14]:
F.ratio_screen(None, 'ROIC', n=TOP_N, ratios_df=ratios, unit=UNIT)

,ticker,company_name,t_q,영업이익,유효세율_%,투하자본(평균),ROIC,OPM,NPM,ROE,ROA,부채비율
0,ANTM,,2026Q1,"156,743.00",15.76,"14,926.00",884.60,78.21,2.61,23.78,4.27,185.70
1,MANH,,2026Q1,281.56,25.46,60.47,347.06,25.58,19.68,96.24,29.91,260.93
2,CVLT,,2026Q1,92.59,23.30,29.49,240.82,7.82,5.97,42.49,4.70,"25,070.87"
3,EXPE,,2026Q1,"2,450.00",18.93,913.00,217.55,16.15,9.81,71.59,5.66,"1,341.12"
4,INOD,,2026Q1,47.08,21.99,19.10,192.28,16.61,13.86,38.60,23.41,64.16
...,...,...,...,...,...,...,...,...,...,...,...,...
95,ROST,,2026Q1,"2,707.36",24.53,"6,633.13",30.80,11.90,9.43,36.68,14.09,151.30
96,HALO,,2026Q1,859.27,30.70,"1,933.96",30.79,56.96,23.13,99.40,14.33,"1,116.89"
97,FTI,,2026Q1,"1,483.00",22.34,"3,748.00",30.73,14.55,10.62,33.37,10.79,199.22
98,TRX,,2026Q1,60.27,35.00,129.25,30.31,46.04,-23.35,-20.81,-13.96,40.86


In [15]:
F.ratio_screen(None, 'OPM',  n=TOP_N, ratios_df=ratios, unit=UNIT)

,ticker,company_name,t_q,매출액,영업이익,OPM,NPM,ROE,ROA,ROIC,부채비율
0,DX,,2026Q1,695.85,900.29,129.38,34.75,11.74,1.45,7.48,794.56
1,AGNC,,2026Q1,"3,376.00","4,062.00",120.32,43.60,13.25,1.37,6.45,876.09
2,NLY,,2026Q1,"6,837.90","7,194.05",105.21,31.96,14.86,1.79,4.88,748.58
3,CON,,2026Q1,"2,232.24","2,104.10",94.26,7.97,45.35,6.34,65.74,536.55
4,COG,,2026Q1,"2,819.00","2,396.00",84.99,59.13,22.07,6.99,15.23,62.97
...,...,...,...,...,...,...,...,...,...,...,...
95,BCE,,2026Q1,"24,702.00","10,732.35",43.45,25.44,30.40,8.17,15.00,244.05
96,WPC,,2026Q1,"1,986.27",860.76,43.34,26.02,6.18,2.91,4.89,117.69
97,BFS,,2026Q1,296.25,128.22,43.28,12.43,7.63,1.72,6.27,355.54
98,EXR,,2026Q1,"3,393.64","1,467.82",43.25,27.82,6.51,3.25,5.03,104.74


In [16]:
F.ratio_screen(None, 'FCF_margin', n=TOP_N, ratios_df=ratios, unit=UNIT)

,ticker,company_name,t_q,매출액,FCF,FCF_margin,OPM,NPM,ROE,ROA,ROIC,부채비율
0,STRK,,2026Q1,490.47,"4,523.87",922.36,-8.25,"-2,482.01",-30.76,-24.80,-0.07,18.92
1,MSTR,,2026Q1,490.47,"4,523.87",922.36,-8.25,"-2,482.01",-30.76,-24.80,-0.07,18.92
2,RC,,2026Q1,327.62,873.42,266.60,-145.44,-155.71,-29.24,-6.26,-4.93,335.86
3,TBPH,,2026Q1,109.78,268.74,244.81,0.08,104.34,50.17,28.08,0.08,62.53
4,COG,,2026Q1,"2,819.00","4,523.00",160.45,84.99,59.13,22.07,6.99,15.23,62.97
...,...,...,...,...,...,...,...,...,...,...,...,...
95,GNL,,2026Q1,472.16,178.09,37.72,21.42,-8.72,-2.37,-0.83,1.78,165.96
96,SKT,,2026Q1,596.62,223.48,37.46,19.94,20.76,18.24,4.66,5.05,306.18
97,JOE,,2026Q1,518.10,193.47,37.34,28.47,21.64,14.81,7.31,8.96,95.99
98,IAG,,2026Q1,"3,422.53","1,276.56",37.30,44.82,29.53,25.62,17.76,27.07,35.38


In [ ]:
F.ratio_screen(None, '부채비율', n=TOP_N, ratios_df=ratios, unit=UNIT, ascending=True)

In [ ]:
ratios[ratios['ticker'].isin(['AAPL', 'MSFT', 'NVDA'])].T